# ASAP8 ROI raw-activity movie

Display one ROI's extracted raw fluorescence inside its mask over a static crop
of the local reference image, then save the result as an MP4.

The voltage summary contains an ROI-mean trace rather than a pixel-resolved
movie, so the ROI is filled uniformly according to its raw fluorescence value.

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import json

import h5py
import imageio_ffmpeg
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.colors import Normalize
from IPython.display import Video, display

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Select one session through the registry

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MOUSE = 852835
SESSION_IDX = -3

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"

registry = VIPSessionRegistry.from_basepath(BASE_PATH)
session_df = registry.sessions(
    subject_ids=[TARGET_MOUSE],
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).sort_values("session_date").reset_index(drop=True)

assets = [registry.resolve_assets(row) for _, row in session_df.iterrows()]
asset = assets[SESSION_IDX]

show_cols = [c for c in [
    "session_id", "subject_id", "session_date", "session_type",
    "dmd1_depth", "dmd2_depth", "quality",
] if c in session_df.columns]

display(session_df[show_cols])
print("Selected:", asset.session_id)

## 2. Resolve paths and choose the ROI/movie settings

In [ ]:
VOLTAGE_QC_PATH = (
    asset.qc_dir
    / "voltage"
    / f"voltage_extraction_qc_{TRACE_VARIANT}.json"
)

with open(VOLTAGE_QC_PATH, "r") as f:
    voltage_qc = json.load(f)

FS = float(voltage_qc["sample_rate_hz"])
SUMMARY_PATH = Path(voltage_qc["summary_mat"])
TRACE_H5_PATH = (
    asset.derived_dir
    / "voltage"
    / f"voltage_session_traces_{TRACE_VARIANT}.h5"
)
MOVIE_DIR = asset.derived_dir / "voltage" / "movies"
MOVIE_DIR.mkdir(parents=True, exist_ok=True)

DMD = 1
ROI = 0
TIME_SEC = (10.0, 20.0)

CROP_SIZE_PX = 241
REFERENCE_CHANNEL = 0
REFERENCE_PERCENTILES = (1, 99.7)
ACTIVITY_PERCENTILES = (1, 99.7)

MOVIE_FPS = 60
PLAYBACK_RATE = 0.25   # 0.25 = four-times slower than real time
FRAME_REDUCER = "mean" # "mean" or "max"
ACTIVITY_CMAP = "inferno"
OVERLAY_ALPHA = 0.9
FIGURE_PIXELS = 720

print(f"Session:     {asset.session_id}")
print(f"Sample rate: {FS:,.3f} Hz")
print(f"Trace:       {TRACE_H5_PATH}")
print(f"Summary:     {SUMMARY_PATH}")

## 3. Load the reference image, ROI mask, and raw fluorescence

In [ ]:
with h5py.File(SUMMARY_PATH, "r") as f:
    n_rois = int(
        np.asarray(f["summary/nAnalysisROIs"][()], dtype=int).reshape(-1)[DMD - 1]
    )

    ref = f["summary/refIM"][DMD - 1, 0]
    reference_image = np.flipud(
        np.asarray(f[ref][()], dtype=np.float32).T
    )

    mask_ref = f["summary/masks"][DMD - 1, 0]
    roi_masks = np.asarray(f[mask_ref][()], dtype=bool)
    roi_masks = roi_masks.transpose(0, 2, 1)[:, ::-1, :]

if reference_image.ndim > 2:
    reference_image = reference_image[:, :, REFERENCE_CHANNEL]

with h5py.File(TRACE_H5_PATH, "r") as f:
    raw_f = f[f"DMD{DMD}/raw_f"]
    trace = np.asarray(
        raw_f[ROI] if raw_f.shape[0] == n_rois else raw_f[:, ROI],
        dtype=np.float32,
    ).squeeze()

    time_sec = np.asarray(
        f[f"DMD{DMD}/timebase_sec"][()],
        dtype=np.float64,
    ).squeeze()

roi_mask = roi_masks[ROI]

print("Reference image:", reference_image.shape)
print("ROI masks:      ", roi_masks.shape)
print("Raw trace:      ", trace.shape)

## 4. Crop around the ROI and reduce the trace to movie frames

In [ ]:
yy, xx = np.nonzero(roi_mask)
cy, cx = int(np.mean(yy)), int(np.mean(xx))

crop_size = min(CROP_SIZE_PX, *reference_image.shape)
half = crop_size // 2
y0 = np.clip(cy - half, 0, reference_image.shape[0] - crop_size)
x0 = np.clip(cx - half, 0, reference_image.shape[1] - crop_size)

ys = slice(y0, y0 + crop_size)
xs = slice(x0, x0 + crop_size)
reference_crop = reference_image[ys, xs]
mask_crop = roi_mask[ys, xs]

t0, t1 = TIME_SEC
window = (time_sec >= t0) & (time_sec < t1)
t = time_sec[window]
x = trace[window]

biological_dt = PLAYBACK_RATE / MOVIE_FPS
frame_edges = np.arange(t0, t1 + biological_dt, biological_dt)
frame_times = (frame_edges[:-1] + frame_edges[1:]) / 2

left = np.searchsorted(t, frame_edges[:-1])
right = np.searchsorted(t, frame_edges[1:])

reducer = np.nanmean if FRAME_REDUCER == "mean" else np.nanmax
frame_values = np.array([
    reducer(x[i0:i1]) if i1 > i0 else np.nan
    for i0, i1 in zip(left, right)
])

ref_limits = np.nanpercentile(reference_crop, REFERENCE_PERCENTILES)
activity_limits = np.nanpercentile(
    frame_values[np.isfinite(frame_values)],
    ACTIVITY_PERCENTILES,
)

print(f"{len(frame_values):,} frames")
print(f"{1000 * biological_dt:.2f} ms biological time per frame")
print(f"{len(frame_values) / MOVIE_FPS:.1f} s rendered duration")

## 5. Preview the crop and selected raw trace

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].imshow(
    reference_crop,
    cmap="gray",
    vmin=ref_limits[0],
    vmax=ref_limits[1],
)
axes[0].contour(
    mask_crop,
    levels=[0.5],
    colors="white",
    linewidths=1.5,
)
axes[0].set_title(f"DMD{DMD} ROI {ROI}")
axes[0].axis("off")

axes[1].plot(t, x, lw=0.6)
axes[1].set(
    xlabel="Time (s)",
    ylabel="Raw fluorescence",
    title=f"{t0:g}–{t1:g} s",
)

fig.tight_layout()
plt.show()

## 6. Render and save the MP4

In [ ]:
mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

dpi = 120
fig = plt.figure(
    figsize=(FIGURE_PIXELS / dpi, FIGURE_PIXELS / dpi),
    dpi=dpi,
    facecolor="black",
)
ax = fig.add_axes([0, 0, 1, 1])

ax.imshow(
    reference_crop,
    cmap="gray",
    vmin=ref_limits[0],
    vmax=ref_limits[1],
    interpolation="nearest",
)

activity_image = np.full(mask_crop.shape, np.nan, dtype=np.float32)
cmap = mpl.colormaps[ACTIVITY_CMAP].copy()
cmap.set_bad((0, 0, 0, 0))

overlay = ax.imshow(
    activity_image,
    cmap=cmap,
    norm=Normalize(*activity_limits),
    alpha=OVERLAY_ALPHA,
    interpolation="nearest",
)

ax.contour(
    mask_crop,
    levels=[0.5],
    colors="white",
    linewidths=1.8,
)

ax.text(
    0.025, 0.975,
    f"{asset.session_id} · DMD{DMD} ROI {ROI}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    color="white",
    fontsize=10,
    bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55},
)

time_label = ax.text(
    0.975, 0.975, "",
    transform=ax.transAxes,
    ha="right",
    va="top",
    color="white",
    fontsize=10,
    family="monospace",
    bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55},
)

ax.axis("off")


def update(frame):
    activity_image[:] = np.nan
    if np.isfinite(frame_values[frame]):
        activity_image[mask_crop] = frame_values[frame]
    overlay.set_data(activity_image)
    time_label.set_text(f"t = {frame_times[frame]:.3f} s")
    return overlay, time_label


animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_values),
    interval=1000 / MOVIE_FPS,
    blit=True,
)

movie_path = MOVIE_DIR / (
    f"{asset.session_id}_DMD{DMD}_ROI{ROI}_"
    f"raw_f_{t0:g}-{t1:g}s_{PLAYBACK_RATE:g}x.mp4"
)

writer = FFMpegWriter(
    fps=MOVIE_FPS,
    codec="libx264",
    extra_args=["-pix_fmt", "yuv420p", "-crf", "18"],
)

animation.save(movie_path, writer=writer)
plt.close(fig)

print("Saved:", movie_path)

In [ ]:
display(Video(str(movie_path), embed=False, html_attributes="controls loop"))